# M2: DNS-BERT + Isolation Forest on CIC (Colab)
Owner: Thierry · Backup: Yehan. Runtime → Change runtime type → **GPU**. Run top to bottom.
Checkpoints go to Google Drive, so a disconnect resumes where it stopped (just rerun all cells).

**Inputs** (put in Drive under `MyDrive/leakmate/`):
- `data/processed/cic/queries.parquet` from the parser (contract schema). Until it exists, the notebook falls back to a synthetic sample (results tagged SMOKE, not for the report).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/leakmate'
BRANCH = 'claude/quirky-dijkstra-kgq1zg'   # change to main once merged
!mkdir -p $DRIVE/checkpoints $DRIVE/results

In [ ]:
%cd /content
!rm -rf leakmate && git clone -q -b $BRANCH https://github.com/TerminallyStoned/leakmate.git
%cd /content/leakmate
!pip -q install -r requirements-m2.txt

In [ ]:
# Tranco top-1M (benign names, ref [18]); cached on Drive
import os
if not os.path.exists(f'{DRIVE}/tranco.csv'):
    !wget -q -O /tmp/tranco.zip https://tranco-list.eu/top-1m.csv.zip && unzip -p /tmp/tranco.zip > $DRIVE/tranco.csv
!mkdir -p data/external && cp $DRIVE/tranco.csv data/external/tranco.csv && head -3 data/external/tranco.csv

In [ ]:
CIC = f'{DRIVE}/data/processed/cic/queries.parquet'
if os.path.exists(CIC):
    DATA, EXP = CIC, 'E2'
else:
    print('CIC parquet not found -> synthetic smoke run (NOT for the report)')
    !python scripts/make_synthetic_sample.py --tranco data/external/tranco.csv --n 20000
    DATA, EXP = 'data/sample/synthetic_queries.parquet', 'SMOKE'
import pandas as pd
df = pd.read_parquet(DATA); print(df.shape); df.label.value_counts()

## 1. DNS-BERT, BPE tokenizer (main model)

In [ ]:
!python -m models.m2_run --model m2 --exp $EXP --data $DATA \
    --tranco data/external/tranco.csv --tranco-n 200000 \
    --tokenizer bpe --epochs 3 --batch-size 512 \
    --ckpt-dir $DRIVE/checkpoints/m2_bpe --save-model models/weights/m2_bpe

## 2. Ablation 1: character tokenizer

In [ ]:
!python -m models.m2_run --model m2 --exp $EXP --tag ablation1_char --data $DATA \
    --tranco data/external/tranco.csv --tranco-n 200000 \
    --tokenizer char --epochs 3 --batch-size 512 \
    --ckpt-dir $DRIVE/checkpoints/m2_char --save-model models/weights/m2_char

## 3. Isolation Forest baseline (benign-only)

In [ ]:
!python -m models.m2_run --model iforest --exp $EXP --data $DATA --save-model models/weights/iforest.joblib

## 4. Quick look (official metrics come from the shared eval module)

In [ ]:
import json, glob, numpy as np
from sklearn.metrics import roc_auc_score
for f in sorted(glob.glob('results/*/*.json')):
    r = json.load(open(f)); y, s = np.array(r['labels']), np.array(r['scores'])
    auc = roc_auc_score(y, s) if 0 < y.sum() < len(y) else float('nan')
    print(f"{r['model']:22s} {r['experiment']:6s} AUC={auc:.4f} train={r['train_time_s']:.0f}s lat={r['latency_ms_per_query']:.2f}ms")

In [ ]:
# copy results + weights to Drive; push results/ to your branch from your laptop
!cp -r results/* $DRIVE/results/ && cp -r models/weights $DRIVE/ && ls -R $DRIVE/results | head